### Importing the weights of the YOLO model trained to find the bounding boxes

In [1]:
# Install gdown if it isn't already there
!pip install gdown -U

import gdown

# Paste YOUR specific File ID here
file_id = '15cKoNkawiQcGTX0XKualw-dIKCdtf2SR'

# Download the file and name it 'best.pt'
print("Downloading YOLO weights...")
gdown.download(id=file_id, output='best.pt', quiet=False)
print("Download complete!")

Downloading...
From: https://drive.google.com/uc?id=15cKoNkawiQcGTX0XKualw-dIKCdtf2SR
To: /content/best.pt
100%|██████████| 6.25M/6.25M [00:00<00:00, 99.5MB/s]


Download complete!


### Installation of paddle ocr

In [2]:
# 1. Wipe out any broken or conflicting packages
!pip uninstall -y paddleocr paddlepaddle paddlepaddle-gpu paddlex

# 2. Install YOLO, Gradio, and the STRICTLY STABLE version of PaddleOCR
!pip install ultralytics gradio paddleocr==2.9.1

# 3. Force-install the stable Paddle engine to prevent driver crashes
!python -m pip install paddlepaddle-gpu==2.6.1

Found existing installation: paddleocr 2.9.1
Uninstalling paddleocr-2.9.1:
  Successfully uninstalled paddleocr-2.9.1
  Using cached paddleocr-2.9.1-py3-none-any.whl.metadata (8.5 kB)
Using cached paddleocr-2.9.1-py3-none-any.whl (544 kB)
  Using cached paddlepaddle_gpu-2.6.1-cp312-cp312-manylinux1_x86_64.whl.metadata (8.6 kB)
  Using cached astor-0.8.1-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached opt_einsum-3.3.0-py3-none-any.whl.metadata (6.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.8/758.8 MB 828.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but y

### Gradio for UI installation

In [3]:
!pip install gradio

### OCR output correction to Indian Liscence Plates format

In [4]:
import cv2
import os
import re
import pandas as pd
import logging
import gradio as gr
from ultralytics import YOLO
from paddleocr import PaddleOCR
from google.colab import drive



# 2. Strict Format Mask Function (Dynamic for any length)
def enforce_strict_plate_format(raw_text):
    clean_text = re.sub(r'[^A-Z0-9]', '', raw_text.upper())
    to_number = {'O':'0', 'D':'0', 'Q':'0', 'I':'1', 'L':'1', 'T':'1', 'Z':'2', 'A':'4', 'S':'5', 'G':'6', 'B':'8'}
    to_letter = {'0':'O', '1':'I', '2':'Z', '4':'A', '5':'S', '6':'G', '8':'B'}

    fixed_text = ""
    for i, char in enumerate(clean_text):
        if i < 2:
            # First 2 are always Letters (State Code)
            fixed_text += to_letter.get(char, char) if char.isdigit() else char
        elif i < 4:
            # Next 2 are always Numbers (RTO Code)
            fixed_text += to_number.get(char, char) if char.isalpha() else char
        elif i >= len(clean_text) - 4:
            # The LAST 4 are always Numbers
            fixed_text += to_number.get(char, char) if char.isalpha() else char
        else:
            # The middle characters (Series) are always Letters
            fixed_text += to_letter.get(char, char) if char.isdigit() else char

    return fixed_text

# 3. New Evaluation Logic (> 12 chars = Flag)
def evaluate_plate(plate_text):
    clean_text = plate_text.replace(" ", "").replace("|", "")
    if clean_text == "UNREADABLE" or clean_text == "NOPLATEDETECTED":
        return "❌ Failed to read."
    elif len(clean_text) > 12:
        return "⚠️ FLAGGED FOR MANUAL CHECK: Detected more than 12 characters."
    else:
        return "✅ Processed (Note: OCR predictions may still not be 100% perfect)."

# 4. Load Models
print("Loading YOLO & PaddleOCR...")
# Load the model directly from the downloaded file
model_path = "best.pt"
yolo_model = YOLO(model_path)
ocr = PaddleOCR(use_angle_cls=False, lang='en', use_gpu=False)
print("Models loaded successfully!")



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/paddle/base/framework.py:688: UserWarning: You are using GPU version Paddle, but your CUDA device is not set properly. CPU device will be used by default.
  warnings.warn(


Loading YOLO & PaddleOCR...
download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 3910/3910 [00:17<00:00, 225.46it/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10000/10000 [00:20<00:00, 487.09it/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2138/2138 [00:17<00:00, 119.87it/s]

[2026/05/06 09:50:18] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_length=25, rec_c

Models loaded successfully!


In [7]:
# 5. Core Processing Logic
def process_single_image(img_rgb):
    results = yolo_model(img_rgb, verbose=False)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    plate_found = False

    extracted_plates = [] # Safely store all plates found in the image

    for r in results:
        for box in r.boxes:
            plate_found = True
            x1, y1, x2, y2 = box.xyxy[0].int().tolist()
            cropped_plate = img_rgb[y1:y2, x1:x2]

            # --- ADVANCED PREPROCESSING FOR OCR ---
            # 1. Convert to Grayscale
            gray_plate = cv2.cvtColor(cropped_plate, cv2.COLOR_RGB2GRAY)

            # 2. Resize (Upscale by 2x using Cubic interpolation for smoother edges)
            resized_plate = cv2.resize(gray_plate, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)

            # 3. Apply CLAHE (Locally enhances contrast to fight shadows and glare)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            contrast_plate = clahe.apply(resized_plate)

            # 4. Slight Gaussian Blur (Removes tiny speckles of dirt or noise)
            # A 3x3 blur is small enough to keep letters sharp but removes "salt and pepper" noise
            final_processed_plate = cv2.GaussianBlur(contrast_plate, (3, 3), 0)

            ### OCR

            paddle_results = ocr.ocr(final_processed_plate, cls=False)

            raw_text = ""
            if paddle_results and paddle_results[0] is not None:
                for line in paddle_results[0]:
                    raw_text += line[1][0]

                final_plate_text = enforce_strict_plate_format(raw_text)
                extracted_plates.append(final_plate_text) # Save the corrected version

                # Draw on image
                cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img_bgr, final_plate_text, (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            else:
                cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(img_bgr, "UNREADABLE", (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # Determine what to show in the UI text box
    if len(extracted_plates) > 0:
        final_output_text = " | ".join(extracted_plates)
    elif plate_found:
        final_output_text = "UNREADABLE"
    else:
        final_output_text = "NO PLATE DETECTED"
        cv2.putText(img_bgr, "NO PLATE DETECTED", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    status = evaluate_plate(final_output_text)
    annotated_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    return annotated_rgb, final_output_text, status

def process_batch(files):
    results_data = []

    for file_obj in files:
        file_path = file_obj.name
        file_name = os.path.basename(file_path)

        img = cv2.imread(file_path)
        if img is None:
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        _, final_plate_text, status = process_single_image(img_rgb)

        results_data.append({
            "Image Name": file_name,
            "Extracted Plate": final_plate_text,
            "Evaluation Status": status
        })

    df = pd.DataFrame(results_data)
    csv_path = "/content/batch_results.csv"
    df.to_csv(csv_path, index=False)

    return df, csv_path
# 7. Build the Gradio UI
gr.close_all()
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🚘 Automatic License Plate Recognition System")
    gr.Markdown("Upload images to detect license plates. If the extracted text exceeds 12 characters, it will be flagged for manual review.")

    with gr.Tabs():
        # TAB 1: Single Image Mode
        with gr.TabItem("Single Image Analyzer"):
            with gr.Row():
                with gr.Column():
                    image_input = gr.Image(label="Upload Car Image")
                    analyze_btn = gr.Button("Analyze Plate", variant="primary")
                with gr.Column():
                    image_output = gr.Image(label="Annotated Result")
                    text_output = gr.Textbox(label="Extracted Plate Number", text_align="center")
                    status_output = gr.Textbox(label="Evaluation Status")

            analyze_btn.click(
                fn=process_single_image,
                inputs=image_input,
                outputs=[image_output, text_output, status_output]
            )

        # TAB 2: Batch / Folder Mode
        with gr.TabItem("Batch/Folder Analyzer"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("Select multiple files from your computer to process them all at once.")
                    file_input = gr.File(file_count="multiple", label="Upload Multiple Images")
                    batch_btn = gr.Button("Process Batch", variant="primary")
                with gr.Column():
                    dataframe_output = gr.Dataframe(label="Batch Results Preview")
                    csv_output = gr.File(label="Download CSV Results")

            batch_btn.click(
                fn=process_batch,
                inputs=file_input,
                outputs=[dataframe_output, csv_output]
            )

# Launch the app right inside Colab!
app.launch(debug=True)

Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7a5c95a211c8efc07d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

[2026/05/06 09:57:58] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.033792734146118164
[2026/05/06 09:57:58] ppocr DEBUG: rec_res num  : 1, elapsed : 0.09830260276794434
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7a5c95a211c8efc07d.gradio.live
